<div align="center">

# Universidad Autónoma de Chihuahua
## Facultad de Ingeniería
### Maestría en Ingeniería en Computación

**Materia:** Machine Learning (MIC1)

**Actividad:** Proyecto Final — Notebook 2: Análisis exploratorio de datos (EDA) y preprocesamiento

**Alumno:** Jonahtan Gabriel González Cardoza
**Matrícula:** 345491

**Catedrática:** Dra. Olanda Prieto Ordaz

**Fecha de elaboración:** 21 de noviembre de 2026

</div>

**Proyecto:** predicción de la situación final de un estudiante (*Dropout*, *Enrolled*, *Graduate*) con el dataset
*Predict Students' Dropout and Academic Success* (UCI 697). Este proyecto reúne, en cinco notebooks reproducibles, el trabajo
de las tres prácticas anteriores (regresión logística del Proyecto F_1, SVM y métodos de ensamble).

**Objetivo de este notebook.** Explorar el dataset **solo con Train**, documentar los problemas relevantes (desbalance, atípicos, faltantes ocultos, multicolinealidad) y justificar con ello las decisiones de preprocesamiento del `Pipeline` que usarán todos los modelos.

# III. Desarrollo (continuación)

In [ ]:
from config import *

# Test NO se carga en este notebook: todo el análisis usa solo Train
X_train, _, y_train, _ = cargar_particion()
numeric_cols, nominal_cols, binary_cols = columnas()
print(f"Train: {X_train.shape[0]:,} estudiantes × {X_train.shape[1]} características")

## Fase 3. Análisis Exploratorio de Datos — únicamente sobre Train

### 6. EDA
A partir de aquí todo el análisis usa **solo Train**. Se crea un DataFrame temporal con `y_train` para graficar.

In [ ]:
train_eda = X_train.copy()
train_eda["target"] = y_train
train_eda["clase"] = pd.Categorical.from_codes(y_train, CLASES)
for k, clase in enumerate(CLASES):                       # indicadores 0/1 de cada clase
    train_eda[f"es_{clase}"] = (y_train == k).astype(int)

# Las listas de columnas por tipo (numéricas / nominales / binarias) vienen de columnas(), calculadas con Train en el notebook 01
print(f"Numéricas: {len(numeric_cols)} | Nominales (códigos): {len(nominal_cols)} | Binarias: {len(binary_cols)}")

#### 6.1 Valores faltantes

In [ ]:
print("NaN en Train:", train_eda[numeric_cols + nominal_cols + binary_cols].isna().sum().sum())

# Revisión de ceros: estudiantes con 0 unidades inscritas en ambos semestres
sin_registro = ((train_eda["Curricular units 1st sem (enrolled)"] == 0)
                & (train_eda["Curricular units 2nd sem (enrolled)"] == 0))
print("\nEstudiantes con 0 unidades inscritas en ambos semestres:", sin_registro.sum())
print("Curso al que pertenecen:", train_eda.loc[sin_registro, "Course"].value_counts().to_dict())
print("Situación final de esos estudiantes:", train_eda.loc[sin_registro, "clase"].value_counts().to_dict())

**Interpretación.**
- **NaN:** Train no tiene valores faltantes, así que no se requiere imputación.
- **Faltantes ocultos:** 148 estudiantes de Train tienen 0 unidades inscritas en ambos semestres, y todos son del curso 171 (*Animation and Multimedia Design*). Como 64 de ellos se titularon (los otros son 63 *Dropout* y 21 *Enrolled*), esos ceros no pueden ser "cero materias cursadas": son un hueco de registro de ese curso.
- **Decisión:** no se imputan. `Course` es una variable del modelo (con one-hot, el curso 171 tiene su propia columna), así que el modelo puede distinguir a esos estudiantes. Se documenta como limitación del dataset.

#### 6.2 Análisis de la variable objetivo

In [ ]:
conteo = pd.Series(np.bincount(y_train), index=CLASES)
print(pd.DataFrame({"n": conteo, "proporción": (conteo / len(y_train)).round(4)}))

ax = conteo.plot(kind="bar", color=COLORES_CLASES, title="Distribución de las clases en Train")
ax.bar_label(ax.containers[0])
ax.margins(y=0.12)
plt.xticks(rotation=0)
plt.ylabel("Número de estudiantes")
guardar_fig("02_distribucion_clases_train")

**Interpretación.** Train tiene 1,767 *Graduate* (49.93%), 1,137 *Dropout* (32.13%) y 635 *Enrolled* (17.94%). Con este desbalance, un modelo que siempre prediga *Graduate* tendría ≈49.9% de accuracy sin reconocer a ningún estudiante de las otras dos clases, y su F1 macro sería de apenas ≈0.22. Por eso:
- se evalúa con Precision, Recall y F1 **macro** y con ROC-AUC uno contra el resto, además de Accuracy;
- se explora `class_weight` en la optimización;
- se analiza la regla de decisión para la clase *Dropout* (Fase 8).

#### 6.3 Distribuciones y posibles valores atípicos

In [ ]:
vars_continuas = ["Admission grade", "Previous qualification (grade)", "Age at enrollment",
                  "Curricular units 1st sem (grade)", "Curricular units 2nd sem (grade)",
                  "Curricular units 2nd sem (approved)"]

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, col in zip(axes.ravel(), vars_continuas):
    sns.histplot(data=train_eda, x=col, hue="clase", hue_order=CLASES, palette=PALETA_CLASES,
                 stat="density", common_norm=False, bins=30, element="step", ax=ax)
    ax.set_title(col)
plt.tight_layout()
guardar_fig("02_distribuciones_y_atipicos")

# Conteo de atípicos con la regla de 1.5 × IQR
def resumen_iqr(s):
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    fuera = ((s < q1 - 1.5 * iqr) | (s > q3 + 1.5 * iqr)).sum()
    return pd.Series({"min": s.min(), "max": s.max(), "atípicos (1.5·IQR)": fuera, "% atípicos": 100 * fuera / len(s)})

train_eda[vars_continuas].apply(resumen_iqr).T.round(2)

**Interpretación.**
- **Notas de admisión y de bachillerato:** distribuciones aproximadamente simétricas entre 95 y 190 (escala 0–200), muy parecidas en las tres clases.
- **Edad:** muy sesgada a la derecha: la mayoría se inscribe entre los 18 y 20 años, con una cola larga de estudiantes mayores en la que predominan los *Dropout*.
- **Promedios del 1er y 2º semestre:** bimodales, con un pico en 0 (estudiantes que no aprobaron nada) que es casi exclusivo de los *Dropout*. Fuera del cero, los *Enrolled* tienen promedios algo menores que los *Graduate* (≈11–12 contra ≈13–14).
- **Unidades aprobadas del 2º semestre:** separan bien las clases: los *Dropout* se concentran en 0, los *Enrolled* entre 3 y 5 y los *Graduate* en 6 (la carga habitual por semestre) o más.

**Valores atípicos (regla 1.5·IQR):** 1.9% en la nota de admisión, 3.8% en la de bachillerato, 10.1% en la edad, 16.7% y 20.1% en los promedios del 1er y 2º semestre, y 1.0% en unidades aprobadas del 2º semestre.

**Decisión: no se elimina ningún valor.** Todos están dentro de rangos válidos y son plausibles:
- Los "atípicos" de edad son estudiantes mayores reales.
- Los de los promedios son en su mayoría los ceros de quienes no aprobaron ninguna materia (o el hueco de registro del curso 171).

Precisamente esos valores son los más informativos para separar a los *Dropout*.

#### 6.4 Relación entre características numéricas y la variable objetivo

In [ ]:
def barras_composicion(grupos, ax, titulo, xlabel):
    """Barras apiladas al 100%: proporción de cada clase dentro de cada grupo."""
    comp = pd.crosstab(grupos, train_eda.loc[grupos.index, "clase"], normalize="index")[CLASES]
    comp.plot(kind="bar", stacked=True, color=COLORES_CLASES, ax=ax, width=0.8,
              edgecolor="white", linewidth=1.5, legend=False)
    ax.set_title(titulo)
    ax.set_xlabel(xlabel)
    ax.set_ylabel("Proporción de estudiantes")
    ax.tick_params(axis="x", rotation=0)
    return comp

fig, axes = plt.subplots(1, 3, figsize=(17, 4.8))
edad_rango = pd.cut(train_eda["Age at enrollment"], bins=[16, 19, 22, 25, 30, 40, 71],
                    labels=["17-19", "20-22", "23-25", "26-30", "31-40", "41+"])
comp_edad = barras_composicion(edad_rango, axes[0], "Clases por edad al inscribirse", "Edad")
print("Composición por rango de edad:")
display(comp_edad.round(3))

for ax, col, titulo in [(axes[1], "Curricular units 1st sem (approved)", "aprobadas 1er sem."),
                        (axes[2], "Curricular units 2nd sem (approved)", "aprobadas 2º sem.")]:
    aprobadas = train_eda.loc[~sin_registro, col].clip(upper=10)
    comp = barras_composicion(aprobadas, ax, f"Clases por unidades {titulo}", "Unidades aprobadas (10 = 10 o más)")
    print(f"Composición por unidades {titulo}:")
    display(comp.round(3))
axes[2].legend(CLASES, loc="upper left", bbox_to_anchor=(1.0, 1.0), title="Clase")
plt.tight_layout()
guardar_fig("02_clases_por_edad_y_unidades")

train_eda.groupby("clase", observed=True)[["Admission grade", "Age at enrollment", "Curricular units 1st sem (approved)",
                                           "Curricular units 2nd sem (approved)", "Curricular units 2nd sem (grade)"]].mean().round(2)

**Interpretación.**
- **Edad.** La proporción de *Graduate* cae de 62.4% entre 17 y 19 años a 24.4% entre 26 y 30, mientras que la de *Dropout* sube de 20.7% a 60.3% y luego baja un poco (55.2% entre 31 y 40; 50.6% a partir de 41). *Enrolled* es más frecuente entre 20 y 25 años (22.4–23.3%) y menos entre los mayores (12–15%). La relación **no es lineal**.
- **Unidades aprobadas.** Es la relación más fuerte del dataset, y **ordena a las tres clases**:
  - con 0 unidades aprobadas en el 2º semestre, 94.3% son *Dropout*;
  - con 3 y 4 aprobadas, *Enrolled* es la clase más frecuente (44.8% y 37.5%);
  - con 6 a 8 aprobadas, 79.7–90.0% son *Graduate*.

  En el 1er semestre el patrón es el mismo, aunque *Enrolled* nunca llega a ser la clase más frecuente. El repunte de *Dropout* a partir de 9 unidades corresponde a grupos más pequeños (28–30 estudiantes con 9 unidades y 132–162 con 10 o más).
- **Medias por clase** (*Dropout* / *Enrolled* / *Graduate*):
  - unidades aprobadas: 2.57 / 4.36 / 6.20 en el 1er semestre y 1.94 / 4.10 / 6.16 en el 2º;
  - promedio del 2º semestre: 5.84 / 11.15 / 12.67;
  - edad: 26.1 / 22.4 / 21.7 años;
  - nota de admisión: 124.8 / 125.5 / 128.7.

**Conclusión.** En las variables académicas *Enrolled* queda **entre** las otras dos clases, y la edad casi no lo distingue de *Graduate*. Por eso se espera que sea la clase más difícil, con confusiones hacia ambos lados.

#### 6.5 Variables binarias y categóricas

In [ ]:
vars_bin = ["Tuition fees up to date", "Debtor", "Scholarship holder", "Daytime/evening attendance", "Gender", "Displaced"]
tabla_bin = pd.concat({col: pd.crosstab(train_eda[col], train_eda["clase"], normalize="index")[CLASES]
                       for col in vars_bin}, names=["Variable", "Valor"])
tabla_bin.insert(0, "n", pd.concat({col: train_eda[col].value_counts().sort_index() for col in vars_bin}).values)
display(tabla_bin.round(3))

NOMBRES_CURSO = {
    33: "Biofuel Production Technologies", 171: "Animation and Multimedia Design",
    8014: "Social Service (evening)", 9003: "Agronomy", 9070: "Communication Design",
    9085: "Veterinary Nursing", 9119: "Informatics Engineering", 9130: "Equinculture",
    9147: "Management", 9238: "Social Service", 9254: "Tourism", 9500: "Nursing",
    9556: "Oral Hygiene", 9670: "Advertising and Marketing Mgmt",
    9773: "Journalism and Communication", 9853: "Basic Education", 9991: "Management (evening)",
}
tabla_curso = pd.crosstab(train_eda["Course"], train_eda["clase"], normalize="index")[CLASES]
tabla_curso.insert(0, "n", train_eda["Course"].value_counts())
tabla_curso = tabla_curso.rename(index=NOMBRES_CURSO).sort_values("Dropout")
display(tabla_curso.round(3).T)

ax = tabla_curso[CLASES].set_axis([f"{c} (n={n})" for c, n in tabla_curso["n"].items()]).plot(
    kind="barh", stacked=True, color=COLORES_CLASES, figsize=(9, 7.5), width=0.8,
    edgecolor="white", linewidth=1.5, title="Composición de clases por curso (Train)")
ax.set_xlabel("Proporción de estudiantes")
ax.legend(title="Clase", loc="upper left", bbox_to_anchor=(1.0, 1.0))
guardar_fig("02_composicion_por_curso")

print("Correlación de Pearson entre el CÓDIGO de Course y el indicador de cada clase:",
      {c: round(train_eda["Course"].corr(train_eda[f"es_{c}"]), 3) for c in CLASES})
print("Categorías por variable nominal:", train_eda[nominal_cols].nunique().to_dict())
guardar_tabla(tabla_bin, "02_variables_binarias")
guardar_tabla(tabla_curso, "02_clases_por_curso")

**Interpretación.**

**Variables binarias.** Las de pago separan muy bien a los *Dropout*: con la colegiatura atrasada, 86.9% abandona. En cambio, casi ninguna binaria cambia mucho la proporción de *Enrolled* (por ejemplo, 17.7% entre deudores contra 18.0% entre no deudores); la excepción es la beca (11.6% contra 20.1%).

| Variable (n en Train) | *Dropout* | *Enrolled* | *Graduate* |
|---|---:|---:|---:|
| Colegiatura **no** al corriente (419) / al corriente (3,120) | 86.9% / 24.8% | 8.1% / 19.3% | 5.0% / 56.0% |
| Deudor (389) / no deudor (3,150) | 64.0% / 28.2% | 17.7% / 18.0% | 18.3% / 53.8% |
| Becario (894) / no becario (2,645) | 12.9% / 38.6% | 11.6% / 20.1% | 75.5% / 41.3% |
| Turno nocturno (382) / diurno (3,157) | 43.5% / 30.8% | 15.7% / 18.2% | 40.8% / 51.0% |
| Hombres (1,241) / mujeres (2,298) | 44.8% / 25.3% | 20.1% / 16.8% | 35.1% / 57.9% |
| Desplazados (1,942) / no desplazados (1,597) | 27.5% / 37.8% | 17.8% / 18.2% | 54.7% / 44.1% |

**Variable nominal `Course`.**
- **Composición por curso:** el *Dropout* va de 14.8% en *Nursing* (n = 607) a 55.7% en *Equinculture* (*Biofuel Production Technologies* marca 70%, pero con solo 10 estudiantes). El *Enrolled* también varía mucho: de 10.3% en *Social Service* a 39.8% en *Informatics Engineering*, donde solo 9.0% se titula a tiempo.
- **El código no sirve como número:** su correlación con el indicador de cada clase es de −0.035, 0.012 y 0.023, aunque el curso sí es informativo, porque el código no tiene orden. Por eso se codifica con one-hot.
- **Categorías raras:** las variables nominales tienen hasta 41 categorías en Train (`Father's occupation`), muchas muy raras. Por eso las categorías con menos de 1% de frecuencia se agrupan en una sola columna.

#### 6.6 Matriz de correlación (variables numéricas y binarias)

In [ ]:
cmap_div = LinearSegmentedColormap.from_list("azul_gris_rojo", ["#2a78d6", "#f0efec", "#e34948"])
indicadores = [f"es_{c}" for c in CLASES]
corr_matrix = train_eda[numeric_cols + binary_cols + indicadores].corr()

plt.figure(figsize=(16, 13))
sns.heatmap(corr_matrix, cmap=cmap_div, vmin=-1, vmax=1, center=0, linewidths=0.3, square=True,
            cbar_kws={"shrink": 0.6})
plt.title("Matriz de correlación (Pearson) — Train, con un indicador 0/1 por clase")
plt.tight_layout()
guardar_fig("02_matriz_correlacion")

# Correlación de cada característica con el indicador de cada clase (punto-biserial)
corr_clases = corr_matrix.loc[numeric_cols + binary_cols, indicadores]
corr_clases.columns = CLASES
top = corr_clases.abs().max(axis=1).sort_values(ascending=False).index[:10]
print("Correlación con el indicador de cada clase (top 10 por magnitud máxima):")
display(corr_clases.loc[top].round(3))
print("Mayores |r| con el indicador de Enrolled:")
print(corr_clases["Enrolled"].sort_values(key=abs, ascending=False).head(3).round(3))

cm_feats = corr_matrix.loc[numeric_cols + binary_cols, numeric_cols + binary_cols]
pares = cm_feats.where(np.triu(np.ones(cm_feats.shape, dtype=bool), k=1)).stack()
pares = pares[pares.abs() > 0.7].sort_values(key=abs, ascending=False)
print(f"\nPares de características con |r| > 0.7: {len(pares)}")
print(pares.round(3).head(6))
guardar_tabla(corr_clases, "02_correlacion_con_cada_clase")

**Interpretación.**
- **Correlación con *Dropout* y *Graduate*.** Las variables con mayor asociación lineal son las de rendimiento académico, con signos opuestos para las dos clases: promedio del 2º semestre (−0.574 con *Dropout*, +0.471 con *Graduate*), unidades aprobadas del 2º semestre (−0.567 / +0.568), aprobadas del 1er semestre (−0.475 / +0.483) y promedio del 1er semestre (−0.482 / +0.412). Les siguen `Tuition fees up to date` (−0.430 / +0.329) y `Scholarship holder` (−0.240 / +0.297). La edad (+0.255 / −0.198) y ser deudor (+0.240 / −0.223) van en sentido contrario.
- **Correlación con *Enrolled*.** Ninguna variable pasa de |r| = 0.17. Las mayores son las **evaluaciones** del 2º y 1er semestre (+0.163 y +0.129): son estudiantes que presentan muchas evaluaciones, pero aprueban menos unidades que los *Graduate*. Que la correlación sea baja no significa que *Enrolled* sea impredecible: como su perfil académico es intermedio, un indicador 0/1 de "estar en medio" no se correlaciona linealmente con ninguna variable. La regresión multinomial sí puede representarlo, porque compara los puntajes de las tres clases: *Enrolled* gana en la zona intermedia, donde los puntajes de *Dropout* y *Graduate* son moderados.
- **Multicolinealidad.** Hay 12 pares de características con |r| > 0.7, todos entre unidades curriculares, sobre todo la misma medida en ambos semestres (acreditadas 0.948, inscritas 0.943, aprobadas 0.907, promedio 0.839). La multicolinealidad no afecta la capacidad predictiva, pero vuelve inestables los coeficientes individuales.
- **Decisión:** no se elimina ninguna variable por correlación. La estabilidad se controla con regularización (se exploran L1 y L2 en la Fase 6).
- **Nominales:** se excluyeron de la matriz porque Pearson sobre códigos no tiene sentido.

#### 6.7 Variables macroeconómicas

In [ ]:
macro_cols = ["Unemployment rate", "Inflation rate", "GDP"]
print("Combinaciones distintas en Train:", train_eda[macro_cols].drop_duplicates().shape[0])
tabla_macro = pd.crosstab([train_eda[c] for c in macro_cols], train_eda["clase"], normalize="index")[CLASES]
tabla_macro.insert(0, "n", train_eda.groupby(macro_cols).size())
tabla_macro.round(3)

**Interpretación.** `Unemployment rate`, `Inflation rate` y `GDP` son indicadores nacionales de Portugal del año de inscripción: en Train solo hay 10 combinaciones, una por cohorte. Entre cohortes, el *Dropout* va de 23.0% a 37.7%, el *Enrolled* de 13.1% a 20.7% y el *Graduate* de 43.1% a 56.4%, sin un patrón claro respecto a la economía. Se conservan, porque no hay razón para excluirlas a priori, pero se espera que aporten poco.

### Pregunta obligatoria: ¿Qué decisiones de preprocesamiento o modelado surgieron a partir del EDA realizado sobre Train?

1. **`OneHotEncoder` para las 9 variables nominales.** Son códigos sin orden: el código de `Course` tiene r ≈ 0 con el indicador de cada clase, aunque el *Dropout* por curso va de 14.8% a 55.7% y el *Enrolled* de 10.3% a 39.8%. La regresión logística, el SVM y los miembros lineales de *voting* y *stacking* lo necesitan; un árbol sobre el código numérico solo podría separar por umbrales (`Course ≤ 9119`), mezclando cursos sin relación entre sí. Se usa `min_frequency = 0.01` porque hay variables con hasta 41 categorías. La EBM, en cambio, trata estas 9 variables como nominales de forma nativa (sin *one-hot*).
2. **`StandardScaler` para las 19 numéricas y `passthrough` para las 8 binarias.** Los árboles y el *boosting* son invariantes a transformaciones monótonas, así que no lo necesitan; pero la regresión logística y el SVM sí (las escalas van de 0–26 unidades a 95–190 en notas y 17–70 años), y los ensambles heterogéneos (*voting*, *stacking*) los mezclan con árboles. Se usa **un solo preprocesador para todos**, para que la comparación sea justa. La EBM recibe las variables originales sin escalar.
3. **Sin imputación.** No hay NaN. Los ceros del curso 171 se documentan como limitación porque el modelo recibe `Course`.
4. **No eliminar valores atípicos.** Son válidos y son justamente los más informativos (estudiantes mayores, estudiantes sin materias aprobadas). Los árboles son robustos a ellos porque dividen por orden, no por magnitud.
5. **Modelos que capturan no linealidades e interacciones como hipótesis a probar.** El EDA mostró dos relaciones que un modelo lineal solo captura como tendencia: el pico de *Dropout* entre 26 y 30 años y la posición intermedia de *Enrolled* en las variables académicas. Los árboles las representan directamente (umbrales e interacciones entre variables); la práctica mide si eso mejora la predicción respecto a la regresión logística.
6. **Multicolinealidad.** Hay 12 pares con |r| > 0.7 entre unidades curriculares del 1er y 2º semestre. No afecta la predicción de los ensambles de árboles, pero **reparte la importancia** entre variables correlacionadas (por eso se compara la importancia por impureza con la de permutación, Sección 29) y la EBM la maneja variable por variable.
7. **Métricas macro y pesos de clase.** *Enrolled* es la clase minoritaria (17.9%) y tiene un perfil intermedio, así que será la más difícil. Las métricas macro le dan el mismo peso que a las demás y se usan pesos de clase `balanced` en los ensambles (en `GradientBoostingClassifier` y en la EBM, que no tienen `class_weight`, se calcula el peso por muestra).

## Fase 4. Preprocesamiento automatizado

### 7. Construcción del Pipeline
Se usa el mismo `ColumnTransformer` del Proyecto F_1 y de la práctica de SVM, para que la comparación con los modelos anteriores sea justa:

| Bloque | Variables | Transformación | Justificación |
|---|---|---|---|
| `num` | 19 numéricas/ordinales | `StandardScaler` | Lo requieren la regresión logística y el SVM; no afecta a los árboles |
| `cat` | 9 nominales (códigos) | `OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=0.01)` | Son códigos sin orden; las categorías con < 1% se agrupan |
| `bin` | 8 binarias | `passthrough` | Ya están en 0/1 |

Estructura: **Pipeline = Preprocesamiento + Modelo**. La función `crear_pipeline()` arma cada pipeline con una copia nueva del preprocesador (`clone`), para que ajustar un modelo no modifique el preprocesador de los demás. La EBM es la excepción: trabaja con las 36 variables originales (sección 15).

In [ ]:
preprocessor = crear_preprocessor()          # definido en config.py; lo usan los notebooks 03 y 04

# Inspección informativa del preprocesador ajustado con Train (no se usa para evaluar)
prep_demo = clone(preprocessor).fit(X_train)
nombres_salida = prep_demo.get_feature_names_out()
print("Columnas de entrada:", X_train.shape[1], "→ columnas de salida:", len(nombres_salida))
print(pd.Series([n.split("__")[0] for n in nombres_salida]).value_counts().to_dict())
preprocessor

**Preguntas**

1. **¿Hace falta escalar para los ensambles de árboles?** No: un árbol decide con umbrales (`x ≤ t`), y escalar no cambia el orden de los estudiantes. Se deja el escalador en el Pipeline porque la regresión logística y el SVM, que participan en *voting* y *stacking*, sí lo necesitan, y así todos los modelos ven la misma representación.
2. **¿Por qué el preprocesador debe ir dentro del Pipeline?** Porque si se ajustara con todo Train antes de la validación cruzada, la media, la desviación estándar y las categorías frecuentes incluirían información de los folds de validación (fuga de información). Dentro del Pipeline se ajustan solo con la parte de entrenamiento de cada fold. Esto es especialmente importante en `StackingClassifier` y `CalibratedClassifierCV`, que hacen su propia validación cruzada interna.
3. **¿Por qué se usa `clone(preprocessor)`?** Para que cada modelo tenga su propio preprocesador. Si varios pipelines compartieran el mismo objeto, ajustar uno modificaría el de los otros.
4. **¿Qué cambia con el *one-hot* para los árboles?** Permite que un árbol aísle un solo curso con una división, en lugar de agrupar códigos vecinos. El costo es que hay 106 columnas y una variable con muchas categorías (`Course`, `Application mode`) se reparte en muchas columnas; por eso, para interpretar los bosques, la importancia se **suma por variable original** (Sección 29). Las fracciones de `max_features` (por ejemplo 0.2) se refieren a esas 106 columnas.
5. **Dimensión:** el preprocesador convierte las 36 columnas en 106 (19 numéricas escaladas, 79 indicadoras *one-hot* y 8 binarias), igual que en el Proyecto F_1.

**Qué produjo este notebook.** Las figuras y tablas del EDA en `outputs/figs/02_*` y `outputs/tablas/02_*`. El preprocesador se construye con
`crear_preprocessor()` (en `config.py`), así que el notebook 03 usa exactamente el mismo.